In [ ]:
# CELLULE 1 - Installation
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'git+https://github.com/ferelking242/zeusdl.git@main#subdirectory=zeusdl'
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'requests'], check=True)
print('OK')

In [ ]:
# CELLULE 2 - Configuration
import os, sys

AGENT_ID     = 'colab-1'
MASTER_URL   = 'https://7262cf76-4386-46c0-a4c6-db2e48981e1f-00-1sts3c4c5faag.kirk.replit.dev'
BOT_TOKEN    = '7909647438:AAFUtLLCo8GOuo8yjgs0C2adGsZeNVBsopw'
CHANNEL      = '3924077577'
OUTPUT_DIR   = '/content/downloads'
QUALITY      = '1080'
WORKERS      = 3
COOKIES_PATH = '/content/bangbros_cookies.txt'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config OK')

In [ ]:
# CELLULE 3a - Upload cookies BangBros
from google.colab import files
import shutil

print('Selectionne ton fichier bangbros_cookies.txt...')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
shutil.move(fname, COOKIES_PATH)

with open(COOKIES_PATH) as f:
    content = f.read()

print('instance_token :', 'OK' if 'instance_token' in content else 'MANQUANT')
print('access_token_ma:', 'OK' if 'access_token_ma' in content else 'MANQUANT')
print('Cookies enregistres dans', COOKIES_PATH)

In [ ]:
# CELLULE 3b - Canal deja configure
print('CHANNEL =', CHANNEL)
print('Le canal est deja configure, tu peux passer a la cellule 4.')

In [ ]:
# CELLULE 4 - Telecharger
import subprocess, os, sys

URL_A_TELECHARGER = 'https://site-ma.bangbros.com/scenes?addon=5971'
FMT_PRESETS = [
    'bv*[height=1080]+ba/b[height=1080]',
    'bv*[height=720]+ba/b[height=720]',
    'bv*[height=480]+ba/b[height=480]',
    'bv*[height=360]+ba/b[height=360]',
    'bv*[vcodec^=av01]+ba/b[height<=1080]',
    'bv*[vcodec^=avc1]+ba/b[height<=1080]',
    'bestvideo+bestaudio/best',
]

cmd_urls = [sys.executable, '-m', 'zeusdl', '--cookies', COOKIES_PATH, '--flat-playlist', '--print', 'url', '--no-warnings', '--quiet', URL_A_TELECHARGER]
print('Recuperation des URLs...')
urls_run = subprocess.run(cmd_urls, capture_output=True, text=True, timeout=120)
urls = [u.strip() for u in urls_run.stdout.splitlines() if u.strip().startswith('http')]
print('URLs trouvees:', len(urls))
for i, u in enumerate(urls[:10], 1):
    print(f'{i}. {u}')

def get_info(url):
    cmd = [sys.executable, '-m', 'zeusdl', '--cookies', COOKIES_PATH, '--dump-single-json', url]
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if p.returncode != 0 or not p.stdout.strip():
        return None
    import json
    return json.loads(p.stdout)

def list_formats(info):
    if not info:
        return []
    out = []
    for f in info.get('formats', []):
        out.append((f.get('format_id'), f.get('vcodec'), f.get('acodec'), f.get('height'), f.get('ext')))
    return out

if not urls:
    print('Aucune URL recuperee.')
else:
    target = urls[:3]
    print('Telechargement test sur', len(target), 'video(s)')
    for url in target:
        print('---')
        print('URL:', url)
        info = get_info(url)
        if info:
            print('Title:', info.get('title'))
            print('ID:', info.get('id'))
            print('Formats disponibles:')
            for fmt_id, vcodec, acodec, height, ext in list_formats(info):
                print(f'- {fmt_id} | {vcodec} | {acodec} | {height}p | {ext}')
        else:
            print('Impossible de recuperer les infos formats.')
        cmd = [sys.executable, '-m', 'zeusdl', '--cookies', COOKIES_PATH, '--no-playlist', '-f', FMT_PRESETS[0], '--merge-output-format', 'mp4', '-o', OUTPUT_DIR + '/%(title)s.%(ext)s', '--concurrent-fragments', str(WORKERS), url]
        print('Download preset:', FMT_PRESETS[0])
        result = subprocess.run(cmd, timeout=120)
        if result.returncode != 0:
            for preset in FMT_PRESETS[1:]:
                print('Retry format:', preset)
                cmd[8] = preset
                result = subprocess.run(cmd, timeout=120)
                if result.returncode == 0:
                    break
        print('RET:', result.returncode)
    videos = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(('.mp4','.mkv','.webm'))]
    print('Fichiers finaux dans le dossier:', len(videos))

In [ ]:
# CELLULE 5 - Envoyer automatiquement vers le canal bangbros
import os, requests

def send_to_channel(folder, bot_token, channel):
    if not channel:
        print('CHANNEL non defini')
        return
    exts = ('.mp4', '.mkv', '.avi', '.mov', '.webm')
    videos = sorted([f for f in os.listdir(folder) if f.lower().endswith(exts)])
    if not videos:
        print('Aucune video dans', folder)
        return
    print(f'{len(videos)} video(s) -> canal {channel}')
    for fname in videos:
        path = os.path.join(folder, fname)
        with open(path, 'rb') as f:
            r = requests.post(
                f'https://api.telegram.org/bot{bot_token}/sendVideo',
                data={'chat_id': channel, 'caption': fname[:1000], 'supports_streaming': 'true'},
                files={'video': (fname, f, 'video/mp4')},
                timeout=600
            )
        print('OK' if r.status_code == 200 else f'Erreur {r.status_code}')

send_to_channel(OUTPUT_DIR, BOT_TOKEN, CHANNEL)

In [ ]:
# CELLULE 6 - Agent Hermes (optionnel)
print('Cellule agent desactivee pour eviter les conflits Telegram.')
print('Lance seulement les cellules 4 et 5.')